<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/iALIGNN_FF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%time
!pip install  -q dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html # for cpu only
!pip install -q git+https://github.com/usnistgov/alignn.git@develop
# !pip install  -q dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html
# Alternate
# !conda install alignn dgl=2.1.0 pytorch torchvision torchaudio pytorch-cuda -c pytorch -c nvidia -y --quiet

1) Example of property prediction model as calculators
They are of two types: `get_figshare_model_ff` (ALIGNN_AtomWise model) and `get_figshare_model_prop` (ALIGNN model)

In [ ]:
from alignn.ff.ff import get_figshare_model_ff,get_figshare_model_prop
from alignn.ff.calculators import AlignnAtomwiseCalculator

dpath = get_figshare_model_ff(model_name="alex_band_gap")
calc_alex_band_gap = AlignnAtomwiseCalculator(path=dpath)

dpath = get_figshare_model_prop(model_name="jv_optb88vdw_bandgap_alignn")
calc_alex_opt_gap = AlignnAtomwiseCalculator(path=dpath)

dpath = get_figshare_model_prop(model_name="jv_mbj_bandgap_alignn")
calc_alex_mbj_gap = AlignnAtomwiseCalculator(path=dpath)

In [ ]:
from jarvis.db.figshare import data
from jarvis.analysis.defects.surface import Surface
from jarvis.core.atoms import Atoms

dft_3d=data('dft_3d')
include_jids = ['JVASP-1180', 'JVASP-30', 'JVASP-1408', 'JVASP-8184', 'JVASP-1183', 'JVASP-62940', 'JVASP-1002', 'JVASP-1174', 'JVASP-8158', 'JVASP-1195', 'JVASP-8003', 'JVASP-1192', 'JVASP-1327', 'JVASP-1372', 'JVASP-1177', 'JVASP-8118']
def get_jid_data(jid=''):
  for i in dft_3d:
    if i['jid']==jid:
      return i
alex=[]
opt=[]
mbj=[]
target=[]
for i in include_jids:
  dat=get_jid_data(i)
  atoms=Atoms.from_dict(dat['atoms'])
  ase_atoms=atoms.ase_converter()
  ase_atoms.calc=calc_alex_band_gap
  alex_band_gap=ase_atoms.get_potential_energy()
  ase_atoms.calc=calc_alex_opt_gap
  opt_band_gap=ase_atoms.get_potential_energy()
  ase_atoms.calc=calc_alex_mbj_gap
  mbj_band_gap=ase_atoms.get_potential_energy()
  print(i,atoms.composition.reduced_formula,alex_band_gap,opt_band_gap,mbj_band_gap,dat['mbj_bandgap'])
  alex.append(alex_band_gap)
  opt.append(opt_band_gap)
  mbj.append(mbj_band_gap)
  target.append(dat['mbj_bandgap'])


In [ ]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(target,alex))
print(mean_absolute_error(target,opt))
print(mean_absolute_error(target,mbj))

3) The following is an integrated calculator i.e. force-field + multi property prediction

In [ ]:
from jarvis.db.figshare import data
from jarvis.db.jsonutils import loadjson,dumpjson
import torch
from alignn.ff.calculators import iAlignnAtomwiseCalculator,AlignnAtomwiseCalculator  # Import from new locationx`
from jarvis.core.atoms import Atoms
import matplotlib.pyplot as plt
from ase.visualize.plot import plot_atoms
import numpy as np
from matplotlib import cm
from matplotlib.gridspec import GridSpec
from jarvis.io.vasp.inputs import Poscar
from tqdm import tqdm
interface_db = data("interfacedb")
atoms = Atoms.from_dict(interface_db[0]["atoms"])
calc = iAlignnAtomwiseCalculator()
ase_atoms = atoms.ase_converter()

ase_atoms.calc=calc
en = ase_atoms.get_potential_energy()

In [ ]:
charges=ase_atoms.get_charges()
norm = plt.Normalize(min(charges), max(charges))
cmap = cm.viridis
colors = cmap(norm(charges))
fig, ax = plt.subplots()
plot_atoms(
    atoms.ase_converter(),
    ax,
    radii=0.5,
    rotation=("90x,45y,0z"),
    colors=colors,
)
fig.colorbar(
    cm.ScalarMappable(norm=norm, cmap=cmap), label="Charge (e)", ax=ax
)  # Colorbar

plt.axis("off")


In [ ]:
mags=ase_atoms.get_magnetic_moments()
norm = plt.Normalize(min(mags), max(mags))
cmap = cm.viridis
colors = cmap(norm(mags))
fig, ax = plt.subplots()
plot_atoms(
    atoms.ase_converter(),
    ax,
    radii=0.5,
    rotation=("90x,45y,0z"),
    colors=colors,
)
fig.colorbar(
    cm.ScalarMappable(norm=norm, cmap=cmap), label="MagMom", ax=ax
)  # Colorbar

plt.axis("off")


In [ ]:
fig, ax = plt.subplots()
plot_atoms(
    atoms.ase_converter(),
    ax,
    radii=0.5,
    rotation=("90x,45y,0z"),
)
plt.axis("off")


In [ ]:
#prop_names = ase_atoms.calc.implemented_properties
prop_vals = ase_atoms.calc.results

In [ ]:
for i in ase_atoms.calc.props:
  print(i,prop_vals[i])

In [ ]:
from jarvis.io.vasp.inputs import Poscar
pos="""O6Al4
1.0
4.253255599774612 -0.0011343765684713 2.94300731057024
1.541858721105255 3.963944319736534 2.943006924432697
-0.0016591490976551 -0.0011346142113319 5.172181923360239
Al O
4 6
Cartesian
3.752634939834702 2.566123983859001 7.162796489886124
4.937556494128162 3.37639665986629 9.424503357100216
0.8559048131793948 0.5852841675191416 1.6336962577356697
2.040826137898389 1.3955567148399188 3.895402322542909
1.9733553931812493 0.22091245126772005 5.350640134181611
4.399319088512879 0.9892904214065957 6.389843718224711
1.394141923908782 2.9723904285592018 4.668355192287564
3.8743295023397937 1.758798762665216 3.803753825907229
1.919131734641725 2.2028820978419414 7.254445356129261
3.820105770814909 3.7407682772449635 5.707558887664717
"""
ase_atoms=Poscar.from_string(pos).atoms.ase_converter()
ase_atoms.calc=calc
en = ase_atoms.get_potential_energy()
prop_vals = ase_atoms.calc.results
for i in ase_atoms.calc.props:
  print(i,prop_vals[i])

In [ ]:
from jarvis.io.vasp.inputs import Poscar
pos="""Bi2.14908 Ca0.2 Cu O4 Sr1.40148
1.0
        5.4899997711         0.0000000000         0.0000000000
        0.0000000000        28.0200004578         0.0000000000
        0.0000000000         0.0000000000        24.8400001526
   Bi    O   Ca   Cu   Sr
   72   80    4   20   36
Direct
     0.750000000         0.200000000         0.050000000
     0.250000000         0.800000000         0.950000000
     0.750000000         0.800000000         0.050000000
     0.250000000         0.200000000         0.950000000
     0.750000000         0.700000000         0.550000000
     0.250000000         0.300000000         0.450000000
     0.750000000         0.300000000         0.550000000
     0.250000000         0.700000000         0.450000000
     0.250000000         0.300000000         0.050000000
     0.750000000         0.700000000         0.950000000
     0.250000000         0.700000000         0.050000000
     0.750000000         0.300000000         0.950000000
     0.250000000         0.800000000         0.550000000
     0.750000000         0.200000000         0.450000000
     0.250000000         0.200000000         0.550000000
     0.750000000         0.800000000         0.450000000
     0.250000000         0.500000000         0.050000000
     0.750000000         0.500000000         0.950000000
     0.250000000         0.000000000         0.550000000
     0.750000000         0.000000000         0.450000000
     0.250000000         0.100000000         0.050000000
     0.750000000         0.900000000         0.950000000
     0.250000000         0.900000000         0.050000000
     0.750000000         0.100000000         0.950000000
     0.250000000         0.600000000         0.550000000
     0.750000000         0.400000000         0.450000000
     0.250000000         0.400000000         0.550000000
     0.750000000         0.600000000         0.450000000
     0.250000000         0.200000000         0.180000000
     0.750000000         0.800000000         0.820000000
     0.250000000         0.800000000         0.180000000
     0.750000000         0.200000000         0.820000000
     0.250000000         0.700000000         0.680000000
     0.750000000         0.300000000         0.320000000
     0.250000000         0.300000000         0.680000000
     0.750000000         0.700000000         0.320000000
     0.250000000         0.400000000         0.180000000
     0.750000000         0.600000000         0.820000000
     0.250000000         0.600000000         0.180000000
     0.750000000         0.400000000         0.820000000
     0.250000000         0.900000000         0.680000000
     0.750000000         0.100000000         0.320000000
     0.250000000         0.100000000         0.680000000
     0.750000000         0.900000000         0.320000000
     0.750000000         0.300000000         0.180000000
     0.250000000         0.700000000         0.820000000
     0.750000000         0.700000000         0.180000000
     0.250000000         0.300000000         0.820000000
     0.750000000         0.800000000         0.680000000
     0.250000000         0.200000000         0.320000000
     0.750000000         0.200000000         0.680000000
     0.250000000         0.800000000         0.320000000
     0.750000000         0.000000000         0.050000000
     0.250000000         0.000000000         0.950000000
     0.750000000         0.500000000         0.550000000
     0.250000000         0.500000000         0.450000000
     0.750000000         0.100000000         0.180000000
     0.250000000         0.900000000         0.820000000
     0.750000000         0.900000000         0.180000000
     0.250000000         0.100000000         0.820000000
     0.750000000         0.600000000         0.680000000
     0.250000000         0.400000000         0.320000000
     0.750000000         0.400000000         0.680000000
     0.250000000         0.600000000         0.320000000
     0.750000000         0.400000000         0.050000000
     0.250000000         0.600000000         0.950000000
     0.750000000         0.600000000         0.050000000
     0.250000000         0.400000000         0.950000000
     0.750000000         0.900000000         0.550000000
     0.250000000         0.100000000         0.450000000
     0.750000000         0.100000000         0.550000000
     0.250000000         0.900000000         0.450000000
     0.250000000         0.400000000         0.050000000
     0.750000000         0.600000000         0.950000000
     0.250000000         0.600000000         0.050000000
     0.750000000         0.400000000         0.950000000
     0.250000000         0.900000000         0.550000000
     0.750000000         0.100000000         0.450000000
     0.250000000         0.100000000         0.550000000
     0.750000000         0.900000000         0.450000000
     0.250000000         0.300000000         0.160000000
     0.750000000         0.700000000         0.840000000
     0.250000000         0.700000000         0.160000000
     0.750000000         0.300000000         0.840000000
     0.250000000         0.800000000         0.660000000
     0.750000000         0.200000000         0.340000000
     0.250000000         0.200000000         0.660000000
     0.750000000         0.800000000         0.340000000
     0.750000000         0.100000000         0.050000000
     0.250000000         0.900000000         0.950000000
     0.750000000         0.900000000         0.050000000
     0.250000000         0.100000000         0.950000000
     0.750000000         0.600000000         0.550000000
     0.250000000         0.400000000         0.450000000
     0.750000000         0.400000000         0.550000000
     0.250000000         0.600000000         0.450000000
     0.250000000         0.100000000         0.160000000
     0.750000000         0.900000000         0.840000000
     0.250000000         0.900000000         0.160000000
     0.750000000         0.100000000         0.840000000
     0.250000000         0.600000000         0.660000000
     0.750000000         0.400000000         0.340000000
     0.250000000         0.400000000         0.660000000
     0.750000000         0.600000000         0.340000000
     0.750000000         0.300000000         0.050000000
     0.250000000         0.700000000         0.950000000
     0.750000000         0.700000000         0.050000000
     0.250000000         0.300000000         0.950000000
     0.750000000         0.800000000         0.550000000
     0.250000000         0.200000000         0.450000000
     0.750000000         0.200000000         0.550000000
     0.250000000         0.800000000         0.450000000
     0.250000000         0.000000000         0.050000000
     0.750000000         0.000000000         0.950000000
     0.250000000         0.500000000         0.550000000
     0.750000000         0.500000000         0.450000000
     0.250000000         0.200000000         0.050000000
     0.750000000         0.800000000         0.950000000
     0.250000000         0.800000000         0.050000000
     0.750000000         0.200000000         0.950000000
     0.250000000         0.700000000         0.550000000
     0.750000000         0.300000000         0.450000000
     0.250000000         0.300000000         0.550000000
     0.750000000         0.700000000         0.450000000
     0.750000000         0.000000000         0.160000000
     0.250000000         0.000000000         0.840000000
     0.750000000         0.500000000         0.660000000
     0.250000000         0.500000000         0.340000000
     0.750000000         0.200000000         0.160000000
     0.250000000         0.800000000         0.840000000
     0.750000000         0.800000000         0.160000000
     0.250000000         0.200000000         0.840000000
     0.750000000         0.700000000         0.660000000
     0.250000000         0.300000000         0.340000000
     0.750000000         0.300000000         0.660000000
     0.250000000         0.700000000         0.340000000
     0.250000000         0.500000000         0.160000000
     0.750000000         0.500000000         0.840000000
     0.250000000         0.000000000         0.660000000
     0.750000000         0.000000000         0.340000000
     0.750000000         0.500000000         0.050000000
     0.250000000         0.500000000         0.950000000
     0.750000000         0.000000000         0.550000000
     0.250000000         0.000000000         0.450000000
     0.750000000         0.400000000         0.160000000
     0.250000000         0.600000000         0.840000000
     0.750000000         0.600000000         0.160000000
     0.250000000         0.400000000         0.840000000
     0.750000000         0.900000000         0.660000000
     0.250000000         0.100000000         0.340000000
     0.750000000         0.100000000         0.660000000
     0.250000000         0.900000000         0.340000000
     0.750000000         0.500000000         0.180000000
     0.250000000         0.500000000         0.820000000
     0.750000000         0.000000000         0.680000000
     0.250000000         0.000000000         0.320000000
     0.750000000         0.200000000         0.260000000
     0.250000000         0.800000000         0.740000000
     0.750000000         0.800000000         0.260000000
     0.250000000         0.200000000         0.740000000
     0.750000000         0.700000000         0.760000000
     0.250000000         0.300000000         0.240000000
     0.750000000         0.300000000         0.760000000
     0.250000000         0.700000000         0.240000000
     0.750000000         0.000000000         0.260000000
     0.250000000         0.000000000         0.740000000
     0.750000000         0.500000000         0.760000000
     0.250000000         0.500000000         0.240000000
     0.250000000         0.100000000         0.260000000
     0.750000000         0.900000000         0.740000000
     0.250000000         0.900000000         0.260000000
     0.750000000         0.100000000         0.740000000
     0.250000000         0.600000000         0.760000000
     0.750000000         0.400000000         0.240000000
     0.250000000         0.400000000         0.760000000
     0.750000000         0.600000000         0.240000000
     0.250000000         0.400000000         0.180000000
     0.750000000         0.600000000         0.820000000
     0.250000000         0.600000000         0.180000000
     0.750000000         0.400000000         0.820000000
     0.250000000         0.900000000         0.680000000
     0.750000000         0.100000000         0.320000000
     0.250000000         0.100000000         0.680000000
     0.750000000         0.900000000         0.320000000
     0.750000000         0.100000000         0.180000000
     0.250000000         0.900000000         0.820000000
     0.750000000         0.900000000         0.180000000
     0.250000000         0.100000000         0.820000000
     0.750000000         0.600000000         0.680000000
     0.250000000         0.400000000         0.320000000
     0.750000000         0.400000000         0.680000000
     0.250000000         0.600000000         0.320000000
     0.750000000         0.300000000         0.180000000
     0.250000000         0.700000000         0.820000000
     0.750000000         0.700000000         0.180000000
     0.250000000         0.300000000         0.820000000
     0.750000000         0.800000000         0.680000000
     0.250000000         0.200000000         0.320000000
     0.750000000         0.200000000         0.680000000
     0.250000000         0.800000000         0.320000000
     0.250000000         0.000000000         0.180000000
     0.750000000         0.000000000         0.820000000
     0.250000000         0.500000000         0.680000000
     0.750000000         0.500000000         0.320000000
     0.250000000         0.200000000         0.180000000
     0.750000000         0.800000000         0.820000000
     0.250000000         0.800000000         0.180000000
     0.750000000         0.200000000         0.820000000
     0.250000000         0.700000000         0.680000000
     0.750000000         0.300000000         0.320000000
     0.250000000         0.300000000         0.680000000
     0.750000000         0.700000000         0.320000000
"""
ase_atoms=Poscar.from_string(pos).atoms.ase_converter()
ase_atoms.calc=calc
en = ase_atoms.get_potential_energy()
prop_vals = ase_atoms.calc.results
for i in ase_atoms.calc.props:
  print(i,prop_vals[i])

In [ ]:
from jarvis.db.figshare import data
from jarvis.analysis.defects.surface import Surface

dft_3d=data('dft_3d')
include_jids = ['JVASP-1180', 'JVASP-30', 'JVASP-1408', 'JVASP-8184', 'JVASP-1183', 'JVASP-62940', 'JVASP-1002', 'JVASP-1174', 'JVASP-8158', 'JVASP-1195', 'JVASP-8003', 'JVASP-1192', 'JVASP-1327', 'JVASP-1372', 'JVASP-1177', 'JVASP-8118']

def get_surface(jid="", index=[]):
    for i in dft_3d:
        if i["jid"] == jid:
            atoms = Atoms.from_dict(i["atoms"])
            break

    surf = Surface(
        atoms,
        indices=index,
        from_conventional_structure=True,
        thickness=16,
        vacuum=12,
    ).make_surface()
    return surf

rel_val = 10
al_cbms=[]
al_vbms=[]
labels=[]
for i in include_jids:
    atoms = get_surface(jid=i,index=[1,1,0])
    ase_atoms=atoms.ase_converter()
    ase_atoms.calc=calc
    en = ase_atoms.get_potential_energy()
    results=ase_atoms.calc.results
    print(i,atoms.composition.reduced_formula, results['gap'], results['mbj_bandgap'],results["optb88vdw_bandgap"])
    print()
    gap=results['mbj_bandgap'] #results['gap']
    vbm=results['vbm']- rel_val #- gap
    cbm=vbm+gap #results['cbm']

    al_cbms.append(cbm)
    al_vbms.append(vbm)

    # gap, formula = get_gap(i)

    # g, lg = Graph.atom_dgl_multigraph(atoms)
    # alignn_cbm = model_cbm([g, lg]).cpu().detach().numpy().tolist()
    # alignn_vbm = model_vbm([g, lg]).cpu().detach().numpy().tolist()
    # #alignn_evac = model_evac([g, lg]).cpu().detach().numpy().tolist()

    # x_vbm = alignn_vbm - rel_val - gap  #
    # x_cbm = x_vbm + gap  # alignn_cbm-10#-alignn_evac-5

    # al_cbms.append(x_cbm)
    # al_vbms.append(x_vbm)
    label=i
    labels.append(label)

In [ ]:
from math import floor

ppi = 100
figw = 1250
figh = 850
plt.close()
fig = plt.figure(figsize=(figw / ppi, figh / ppi), dpi=ppi)
ax = fig.add_subplot(1,1, 1)
plt.rcParams.update({"font.size": 18})
ax = fig.add_subplot()
x = np.arange(len(al_vbms)) + 0.5
emin_tmp = floor(min(al_vbms)) - 1.0
emin = emin_tmp
y = np.array(al_vbms) - emin
width = 0.9
ax.bar(x, y, bottom=emin, color="skyblue", width=width, align="edge")
ax.set_xticklabels([])
y = -np.array(al_cbms)
# ax.bar(x, y, bottom=cbms,color='lightgreen',width=width,align='edge')
ax.bar(x, y, bottom=al_cbms, color="lightgreen", width=width, align="edge")
ax.set_xlim(0.4, len(labels) + 0.4)
ax.set_ylim(emin_tmp, 0)


ax.set_title("iALIGNN-FF-band alignment")
ax.set_xticklabels([])

ax.set_xticks(np.arange(len(labels)) + 1, labels)
ax.set_xticklabels(labels, rotation=90)
ax.axhline(y=-4.5, linestyle="-.", color="black")
ax.axhline(y=-5.73, linestyle="-.", color="black")
ax.text(max(x) + 1, -4, "${H^+}/{H_2}$")
ax.text(max(x) + 1, -6.2, "${O_2}/{H_2O}$")
# plt.title("2Positions of VBM and CBM ")
# ax.set_ylabel('Energy wrt vacuum (eV)')
# ax.set_xlabel(r'$\leftarrow$2D materials$\rightarrow $')
# ax.set_ylabel(r'$\Delta \Theta / \omega $ \Huge{$\longleftarrow$}')
count = 0
for i, j in zip(al_cbms, al_vbms):
    count += 1
    plt.text(count - 0.5, emin_tmp + 0.1, round(-1 * j, 1), fontsize=8)
    plt.text(count - 0.5, -0.5, round(-1 * i, 1), fontsize=8)
ax.set_yticks([])



plt.tight_layout()
plt.show()